In [1]:
import pandas as pd
import os
from glob import glob
import warnings
import re

In [2]:
# 关闭 openpyxl 的无关提醒
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# ====================== 基本配置 ======================
folder_path = "raw_data/Weight/"  # 替换为你的本地路径

# 先拿到所有 xlsx 文件
all_files = glob(os.path.join(folder_path, "*.xlsx"))

# 排除 Excel 的临时锁文件（以 ~$ 开头的）
file_list = [
    fp for fp in all_files
    if not os.path.basename(fp).startswith("~$")
]

print(f"📂 检测到有效文件数量: {len(file_list)}")
for f in file_list:
    print("  -", os.path.basename(f))


# ====================== 核心函数 ======================
def extract_cleaned_data_from_file(file_path: str) -> pd.DataFrame | None:
    """从单个 Excel 文件中提取并清洗数据，返回纵向 DataFrame。"""
    try:
        # 显式指定 engine，避免环境里自动识别出问题
        xls = pd.ExcelFile(file_path, engine="openpyxl")

        # 只保留以 Sheet 开头的 sheet
        sheet_names = [s for s in xls.sheet_names if s.startswith("Sheet")]
        if not sheet_names:
            print(f"⚠️ 跳过：无 Sheet 页 -> {file_path}")
            return None

        frames = []

        for sheet in sheet_names:
            try:
                # 先读原始表头区，用于取元信息
                df = pd.read_excel(xls, sheet_name=sheet, header=None, engine="openpyxl")

                # 提取元信息（多一点容错）
                frequency  = df.iloc[4, 2] if pd.notna(df.iloc[4, 2]) else df.iloc[4, 0]
                product    = df.iloc[5, 2] if pd.notna(df.iloc[5, 2]) else df.iloc[5, 0]
                flow       = df.iloc[6, 2] if pd.notna(df.iloc[6, 2]) else df.iloc[6, 0]
                indicators = df.iloc[7, 2] if pd.notna(df.iloc[7, 2]) else df.iloc[7, 0]

                # 从第 10 行开始读正式数据（header=9，对应 Excel 第 10 行）
                df_data = pd.read_excel(xls, sheet_name=sheet, header=9, engine="openpyxl")

                # 结构校验
                if df_data.shape[0] < 2 or df_data.shape[1] < 2:
                    print(f"⚠️ 跳过 sheet（结构异常）: {sheet} in {file_path}")
                    continue

                # 删除无效第二行（常为冒号）
                df_data = df_data.drop(index=0).reset_index(drop=True)

                # 标准化前两列列名
                df_data.columns.values[0:2] = ["country", "date"]

                # 添加元信息
                df_data["Sheet"] = sheet
                df_data["SourceFile"] = os.path.basename(file_path)
                df_data["Frequency"] = frequency
                df_data["PRODUCT"] = product
                df_data["FLOW"] = flow
                df_data["INDICATORS"] = indicators

                frames.append(df_data)

            except Exception as e:
                print(f"❌ Sheet 跳过: {sheet} in {file_path}\n{e}")
                continue

        if frames:
            return pd.concat(frames, ignore_index=True)
        else:
            print(f"⚠️ 文件跳过（无有效 sheet 数据）: {file_path}")
            return None

    except Exception as e:
        print(f"❌ 文件读取失败: {file_path}\n{e}")
        return None


# ====================== 批量处理所有文件 ======================
df_list = [extract_cleaned_data_from_file(fp) for fp in file_list]
df_list = [df for df in df_list if df is not None]

if df_list:
    # 合并所有文件的数据
    all_data = pd.concat(df_list, ignore_index=True)

    # 宽转长：国家 + 日期 + 元信息 保持不变，其余列转成 partner / value
    metadata_cols = [
        "country", "date",
        "Sheet", "SourceFile",
        "Frequency", "PRODUCT", "FLOW", "INDICATORS"
    ]
    value_cols = [col for col in all_data.columns if col not in metadata_cols]

    df_long = all_data.melt(
        id_vars=metadata_cols,
        value_vars=value_cols,
        var_name="partner",
        value_name="weight"
    )

    # 替换冒号为空值，并转为 float
    df_long["weight"] = pd.to_numeric(
        df_long["weight"].replace({":": None}),
        errors="coerce"
    )

    # 可选：保存结果
    # df_long.to_csv("combined_cleaned_long.csv", index=False)
    # df_long.to_excel("combined_cleaned_long.xlsx", index=False)

    print("✅ 合并后的长表预览：")
    print(df_long.head())
    print(f"✅ 最终行数: {len(df_long)}")

else:
    print("❗ 所有文件均未成功读取或无有效数据。")


📂 检测到有效文件数量: 26
  - data_2022.xlsx
  - data_2018.xlsx
  - data_2025_07_09_870911_19.xlsx
  - data_2025_01_06_8716.xlsx
  - data_2019.xlsx
  - data_2023.xlsx
  - data_2023_8716.xlsx
  - data_2024_8716.xlsx
  - data_2024.xlsx
  - data_2021_2024_870911_19.xlsx
  - data_2017_8716.xlsx
  - data_2025_01_06_870911_19.xlsx
  - data_2022_8716.xlsx
  - data_2025_07_09_8716.xlsx
  - data_2025_12.xlsx
  - data_2019_8716.xlsx
  - data_2025_01_06.xlsx
  - data_2021_8716.xlsx
  - data_2025_10_11.xlsx
  - data_2020.xlsx
  - data_2017.xlsx
  - data_2018_8716.xlsx
  - data_2020_8716.xlsx
  - data_2021.xlsx
  - data_2025_07_09.xlsx
  - data_2017_2020_870911_19.xlsx
✅ 合并后的长表预览：
   country     date    Sheet      SourceFile Frequency  \
0  Austria  2022-01  Sheet 1  data_2022.xlsx   Monthly   
1  Austria  2022-02  Sheet 1  data_2022.xlsx   Monthly   
2  Austria  2022-03  Sheet 1  data_2022.xlsx   Monthly   
3  Austria  2022-04  Sheet 1  data_2022.xlsx   Monthly   
4  Austria  2022-05  Sheet 1  data_2022.xls

In [3]:
df_long = df_long[['country', 'date', 'PRODUCT', 'INDICATORS', 'partner', 'weight']]

In [4]:
def clean_label(value):
    if pd.isna(value):
        return value
    value = re.sub(r"\s*\(.*", "", value)                       # 括号内容
    value = re.sub(r"\s*from\s+\d{4}.*?\)$", "", value)         # from 1994 -> 2000)
    value = re.sub(r"\s*->\s*\d{4}\)?$", "", value)             # -> 1994)
    return value.strip()

In [5]:
# 删除括号及其内容：Belgium (incl. Luxembourg 'LU' -> 1998) → Belgium
df_long["country"] = df_long["country"].apply(clean_label)

df_long["partner"] = df_long["partner"].apply(clean_label)

# date
df_long["date"] = pd.to_datetime(df_long["date"], errors="coerce", format="%Y-%m")
df_long["year"] = df_long["date"].dt.year
df_long["month"] = df_long["date"].dt.month

In [6]:
# 仅删除 value 列中为 NaN 的行
df_long = df_long.dropna(subset=['weight'])

In [7]:
df_long["date"].unique()

<DatetimeArray>
['2022-05-01 00:00:00', '2022-11-01 00:00:00', '2022-09-01 00:00:00',
 '2022-10-01 00:00:00', '2022-06-01 00:00:00', '2022-12-01 00:00:00',
 '2022-07-01 00:00:00', '2022-08-01 00:00:00', '2022-02-01 00:00:00',
 '2022-04-01 00:00:00',
 ...
 '2017-10-01 00:00:00', '2021-09-01 00:00:00', '2021-02-01 00:00:00',
 '2021-11-01 00:00:00', '2021-01-01 00:00:00', '2021-12-01 00:00:00',
 '2021-07-01 00:00:00', '2025-08-01 00:00:00', '2025-07-01 00:00:00',
 '2020-04-01 00:00:00']
Length: 108, dtype: datetime64[ns]

In [8]:
hs_code_map_87 = {
    "Tractors (other than tractors of heading 8709)": "8701",
    "Motor vehicles for the transport of >= 10 persons, incl. driver": "8702",
    "Motor cars and other motor vehicles principally designed for the transport of <10 persons, incl. station wagons and racing cars (excl. motor vehicles of heading 8702)": "8703",
    "Motor vehicles for the transport of goods, incl. chassis with engine and cab": "8704",
    "Special purpose motor vehicles (other than those principally designed for the transport of persons or goods), e.g. breakdown lorries, crane lorries, fire fighting vehicles, concrete-mixer lorries, road sweeper lorries, spraying lorries, mobile workshops and mobile radiological units": "8705",
    "Chassis fitted with engines, for tractors, motor vehicles for the transport of ten or more persons, motor cars and other motor vehicles principally designed for the transport of persons, motor vehicles for the transport of goods and special purpose motor vehicles of heading 8701 to 8705 (excl. those with engines and cabs)": "8706",
    "Works trucks, self-propelled, not fitted with lifting or handling equipment, of the type used in factories, warehouses, dock areas or airports for short distance transport of goods; tractors of the type used on railway station platforms; parts of the foregoing vehicles, n.e.s.": "8709",
    # 8716 (trailers / semi-trailers / other non-mechanically propelled vehicles)
    "Trailers and semi-trailers; other vehicles, not mechanically propelled (excl. railway and tramway vehicles); parts thereof, n.e.s.": "8716",
    "Trailers and semi-trailers of the caravan type, for housing or camping": "871610",
    "Self-loading or self-unloading trailers and semi-trailers for agricultural purposes": "871620",
    "Tanker trailers and tanker semi-trailers, not designed for running on rails": "871631",
    "Trailers and semi-trailers for the transport of goods, not designed for running on rails (excl. self-loading or self-unloading trailers and semi-trailers for agricultural purposes and tanker trailers and tanker semi-trailers)": "871639",
    "Trailers and semi-trailers, not designed for running on rails (excl. trailers and semi-trailers for the transport of goods and those of the caravan type for housing or camping)": "871640",
    "Vehicles pushed or drawn by hand and other vehicles not mechanically propelled (excl. trailers and semi-trailers)": "871680",
    "Electrical vehicles not fitted with lifting or handling equipment, of the type used in factories, warehouses, dock areas or airports for short distance transport of goods; tractors of the type used on railway station platforms": "870911",
    "Works trucks, self-propelled, not fitted with lifting or handling equipment, of the type used in factories, warehouses, dock areas or airports for short distance transport of goods; tractors of the type used on railway station platforms (excl. electrical trucks)": "870919",
    "Parts of self-propelled works trucks, not fitted with lifting or handling equipment, of the type used in factories, warehouses, dock areas or airports for short distance transport of goods, incl. tractors for railways station platforms, n.e.s.": "870990",
}

df_long["HS Code"] = df_long["PRODUCT"].map(hs_code_map_87)

In [9]:
exclude_values = [
    'All countries of the world',
    'Extra-EU27', 
    'Intra-EU27',
    'Stores and provisions',
    'Stores and provisions within the framework of intra-Union trade',
    'Stores and provisions within the framework of extra-Union trade',
    'Countries and territories not specified',
    'Countries and territories not specified within the framework of intra-Union trade',
    'Countries and territories not specified within the framework of extra-Union trade',
    'Countries and territories not specified for commercial or military reasons',
    'Countries and territories not specified for commercial or military reasons in the framework of intra-Union trade',
    'Countries and territories not specified for commercial or military reasons in the framework of extra-Union trade'
]

df_long = df_long[~df_long['partner'].isin(exclude_values)]

In [10]:
df_long['partner'].unique()

array(['Andorra', 'United Arab Emirates', 'Afghanistan',
       'Antigua and Barbuda', 'Albania', 'Armenia', 'Angola',
       'Antarctica', 'Argentina', 'American Samoa', 'Austria',
       'Australia', 'Aruba', 'Azerbaijan', 'Bosnia and Herzegovina',
       'Barbados', 'Bangladesh', 'Belgium', 'Burkina Faso', 'Bulgaria',
       'Bahrain', 'Burundi', 'Benin', 'Saint Barthélemy', 'Bermuda',
       'Brunei Darussalam', 'Bolivia, Plurinational State of',
       'Bonaire, Sint Eustatius and Saba', 'Brazil', 'Bahamas',
       'Bouvet Island', 'Botswana', 'Belarus', 'Belize', 'Canada',
       'Cocos Islands', 'Congo, Democratic Republic of',
       'Central African Republic', 'Congo', 'Switzerland',
       'Côte d’Ivoire', 'Cook Islands', 'Chile', 'Cameroon', 'China',
       'Colombia', 'Costa Rica', 'Cuba', 'Cabo Verde', 'Curaçao',
       'Christmas Island', 'Cyprus', 'Czechia', 'Germany', 'Djibouti',
       'Denmark', 'Dominica', 'Dominican Republic', 'Algeria', 'Ecuador',
       'Estonia',

In [11]:
df_long['HS Code'].unique()

array(['8701', '8702', '8703', '8704', '8705', '8709', '8716', '871610',
       '871639', '871640', '871620', '871680', '8706', '870919', '871631',
       '870911', '870990'], dtype=object)

In [12]:
hscode_list= ['8709', '8716','8706']
df_long= df_long[~df_long['HS Code'].isin(hscode_list)]

In [13]:
mapping = [
    ("8701", "Specialty"),
    ("8702", "TBR"),
    ("8703", "PCR"),
    ("8704", "TBR"),
    ("8705", "Specialty"),
    ("870911", "Specialty"),
    ("870919", "Specialty"),
    ("871610", "Other"),
    ("871620", "Specialty"),
    ("871631", "TBR"),
    ("871639", "TBR"),
    ("871640", "TBR"),
    ("871680", "Other"),
]

map_dict = dict(mapping)

# 1) 精确匹配（HS Code 列就是这些 4/6 位时推荐）
df_long["General_Category"] = (
    df_long["HS Code"].astype(str).str.strip().map(map_dict).fillna("Other")
)

In [14]:
df_long[['General_Category', 'HS Code']].drop_duplicates()

,General_Category,HS Code
64,Specialty,8701
439,TBR,8702
685,PCR,8703
1113,TBR,8704
1452,Specialty,8705
5151,Other,871610
5652,TBR,871639
5826,TBR,871640
11666,Specialty,871620
12997,Other,871680


In [15]:
df_long = df_long[df_long['General_Category']!='Other']

In [16]:
df_long

,country,date,PRODUCT,INDICATORS,partner,weight,year,month,HS Code,General_Category
64,Germany,2022-05-01,Tractors (other than tractors of heading 8709),QUANTITY_IN_100KG,Andorra,260.00,2022.0,5.0,8701,Specialty
70,Germany,2022-11-01,Tractors (other than tractors of heading 8709),QUANTITY_IN_100KG,Andorra,130.30,2022.0,11.0,8701,Specialty
104,Spain,2022-09-01,Tractors (other than tractors of heading 8709),QUANTITY_IN_100KG,Andorra,85.30,2022.0,9.0,8701,Specialty
105,Spain,2022-10-01,Tractors (other than tractors of heading 8709),QUANTITY_IN_100KG,Andorra,75.64,2022.0,10.0,8701,Specialty
124,France,2022-05-01,Tractors (other than tractors of heading 8709),QUANTITY_IN_100KG,Andorra,131.90,2022.0,5.0,8701,Specialty
...,...,...,...,...,...,...,...,...,...,...
13255288,Netherlands,2017-10-01,"Motor vehicles for the transport of goods, inc...",QUANTITY_IN_100KG,Zimbabwe,76.50,2017.0,10.0,8704,TBR
13255486,France,2017-01-01,Special purpose motor vehicles (other than tho...,QUANTITY_IN_100KG,Zimbabwe,170.00,2017.0,1.0,8705,Specialty
13261822,Belgium,2021-04-01,Motor cars and other motor vehicles principall...,QUANTITY_IN_100KG,Zimbabwe,45.40,2021.0,4.0,8703,PCR
13261826,Belgium,2021-08-01,Motor cars and other motor vehicles principall...,QUANTITY_IN_100KG,Zimbabwe,18.20,2021.0,8.0,8703,PCR


In [17]:
df_long.to_csv('./Data/vehicle_product_weight.csv',index=False)